# Analyse de sentiment des avis d'hôtels TripAdvisor
Objectif : prédire la note (1 à 5 étoiles) d'un avis à partir de son texte, et comprendre ce qui rend les clients mécontents.

## 0. Chargement et vérification des données

In [21]:
from datasets import load_dataset

ds = load_dataset("nhull/tripadvisor-split-dataset-v2")

### Les splits

In [ ]:
#combien il y a de splits, et combien de lignes dans chacun
ds

DatasetDict({
    train: Dataset({
        features: ['review', 'label'],
        num_rows: 30400
    })
    validation: Dataset({
        features: ['review', 'label'],
        num_rows: 1600
    })
    test: Dataset({
        features: ['review', 'label'],
        num_rows: 8000
    })
})

Il y a 40 000 avis en tout (30 400 train, 1 600 validation, 8 000 test), alors que la description du dataset en annonce 80 000.

### Conversion en DataFrames

In [ ]:
df_train = ds["train"].to_pandas()
df_val = ds["validation"].to_pandas()
df_test = ds["test"].to_pandas()

df_train.head()
#df_train.shape

,review,label
0,"Good room, dodgy area\nNothing at all wrong wi...",3.0
1,Great Hotel\nStayed three days over the 4th of...,4.0
2,"Online rating was 3-star, I would give it 1-st...",1.0
3,Great place to stay on PB!\nThis is the perfec...,5.0
4,"Good Value, Great location\nThe Ramada Gaslamp...",4.0


In [24]:
print(df_train["review"].iloc[0])

Good room, dodgy area
Nothing at all wrong with the hotel. Good staff with large rooms and comfortable beds. The location is not ideal and you have to walk through a lot of human detrius to get into town. We thought the tarriff was high considering the area. We later stayed in San francisco in a cheaper hotel in a better location.


### Équilibre des classes

In [ ]:
#les classes sont-elles vraiment équilibrées dans chaque split? ;
print(df_train['label'].value_counts().sort_index())
print(df_val['label'].value_counts().sort_index())
print(df_test['label'].value_counts().sort_index())

label
1.0    6080
2.0    6080
3.0    6080
4.0    6080
5.0    6080
Name: count, dtype: int64
label
1.0    320
2.0    320
3.0    320
4.0    320
5.0    320
Name: count, dtype: int64
label
1.0    1600
2.0    1600
3.0    1600
4.0    1600
5.0    1600
Name: count, dtype: int64


Chaque note représente exactement 20 % des avis dans chaque split (6 080 avis par note dans le train, 320 en validation, 1 600 dans le test). Les classes sont donc équilibrées et il n’y a pas besoin de rééchantillonnage (resampling) pour l’entraînement.

Mais il faut noter que cet équilibre est artificiel : sur TripAdvisor, la majorité des avis sont à 4 ou 5 étoiles et les avis à 1 étoile sont moins courantes (dans la première version de ce projet, 74 % des avis étaient positifs). → Les performances mesurées sur ce test équilibré seront probablement un peu optimistes : en conditions réelles, comme les avis négatifs sont plus rares, une plus grande part des avis signalés comme négatifs seront de fausses alertes.

### Type des colonnes

In [ ]:
df_val.info()

<class 'pandas.DataFrame'>
RangeIndex: 1600 entries, 0 to 1599
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   review  1600 non-null   str    
 1   label   1600 non-null   float64
dtypes: float64(1), str(1)
memory usage: 1.5 MB


In [27]:
#on vérifie qu'ils sont bien unique'
valeur_unique_train = df_train["label"].unique()
valeur_unique_val = df_val["label"].unique()
valeur_unique_test = df_test["label"].unique()

print(valeur_unique_train, valeur_unique_val, valeur_unique_test)

[3. 4. 1. 5. 2.] [5. 1. 2. 3. 4.] [4. 1. 2. 3. 5.]


In [ ]:
df_train["label"] = df_train["label"].astype(int)
df_val["label"] = df_val["label"].astype(int)
df_test["label"] = df_test["label"].astype(int)
print(df_train["label"].dtype, df_val["label"].dtype, df_test["label"].dtype)

dtype('int64')

Label était stocké en float (1.0 à 5.0), alors qu’une note ne prend que des valeurs entières. Après vérification avec .unique(), je l’ai converti en int : c’est plus lisible.